In [1]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    sentencepiece \
    evaluate \
    rouge-score \
    textstat \
    bert-score \
    sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 96.0 MB/s eta 0:00:00


In [2]:
import os
import gc
import time
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    total_vram = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print("Total VRAM:", round(total_vram, 2), "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


In [3]:
asset = load_dataset("facebook/asset")

print(asset)

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

simplification/validation-00000-of-00001(…): reconstructing file:   0%|          |  0.00B /  885kB            

simplification/validation-00000-of-00001(…): downloading bytes:           |  0.00B            

simplification/test-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  170kB            

simplification/test-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/359 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['original', 'simplifications'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['original', 'simplifications'],
        num_rows: 359
    })
})


In [4]:
asset_df = pd.concat(
    [
        asset["validation"].to_pandas(),
        asset["test"].to_pandas()
    ],
    ignore_index=True
)

print("Unique source sentences:", len(asset_df))

print(
    "Total human references:",
    asset_df["simplifications"].apply(len).sum()
)

Unique source sentences: 2359
Total human references: 23590


In [5]:
assert len(asset_df) == 2359
assert asset_df["simplifications"].apply(len).eq(10).all()

print("✅ ASSET verification passed.")
print("Sources:", len(asset_df))
print("References:", asset_df["simplifications"].apply(len).sum())

✅ ASSET verification passed.
Sources: 2359
References: 23590


In [9]:
from huggingface_hub import login

login()

In [10]:
MODEL_NAME = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("✅ Gemma loaded.")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✅ Gemma loaded.


In [11]:
def zero_shot_prompt(sentence):

    return f"""Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Original sentence:
{sentence}

Simplified sentence:"""

In [13]:
!pip install -q sacremoses
import evaluate
import textstat

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ All metrics loaded successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 15.0 MB/s eta 0:00:00


✅ All metrics loaded successfully.


In [14]:
import evaluate
import textstat

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ All metrics loaded successfully.")

✅ All metrics loaded successfully.


In [30]:
def evaluate_one_sample(
    source,
    prediction,
    references
):

    prediction = str(prediction).strip()

    # =====================================
    # EMPTY PREDICTION SAFETY
    # =====================================

    if not prediction:
        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }


    # =====================================
    # SARI
    # =====================================

    sari_result = sari_metric.compute(
        sources=[source],
        predictions=[prediction],
        references=[references]
    )

    sari = sari_result["sari"]


    # =====================================
    # BLEU
    # =====================================

    try:

        bleu_result = bleu_metric.compute(
            predictions=[prediction],
            references=[references],
            smooth=True
        )

        bleu = bleu_result["bleu"]

    except (ZeroDivisionError, ValueError):

        bleu = 0.0


    # =====================================
    # ROUGE-L
    # Mean across all 10 references
    # =====================================

    rouge_scores = []

    for reference in references:

        rouge_result = rouge_metric.compute(
            predictions=[prediction],
            references=[reference]
        )

        rouge_scores.append(
            rouge_result["rougeL"]
        )

    rouge_l = float(
        np.mean(rouge_scores)
    )


    # =====================================
    # BERTScore
    # Mean across all 10 references
    # =====================================

    bert_result = bertscore_metric.compute(
        predictions=[prediction] * len(references),
        references=references,
        lang="en"
    )

    bertscore_f1 = float(
        np.mean(bert_result["f1"])
    )


    # =====================================
    # FKGL
    # =====================================

    try:

        fkgl = textstat.flesch_kincaid_grade(
            prediction
        )

    except Exception:

        fkgl = 0.0


    # =====================================
    # RETURN
    # =====================================

    return {
        "sari": float(sari),
        "bleu": float(bleu),
        "rougeL": rouge_l,
        "bertscore_f1": bertscore_f1,
        "fkgl": float(fkgl)
    }

In [16]:
example = asset_df.iloc[0]

source = example["original"]

prompt = zero_shot_prompt(source)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

# Reset VRAM measurement
if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()

with torch.no_grad():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

end_time = time.perf_counter()

inference_time = (
    end_time - start_time
)

# Remove prompt tokens
generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

# Peak VRAM from generation only
if torch.cuda.is_available():

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

else:

    peak_vram = None


print("SOURCE:")
print(source)

print("\nGEMMA:")
print(prediction)

print(
    "\nInference time:",
    round(inference_time, 3),
    "seconds"
)

print(
    "Peak VRAM:",
    round(peak_vram, 3),
    "GB"
)

SOURCE:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

GEMMA:
Marin, Mendocino, Lake, Napa, Solano, and Contra Costa are adjacent counties.

Inference time: 3.705 seconds
Peak VRAM: 4.892 GB


In [17]:
test_scores = evaluate_one_sample(
    source,
    prediction,
    example["simplifications"]
)

print("\nMETRICS")
print("=" * 50)

for metric, value in test_scores.items():

    print(
        f"{metric}: {value:.4f}"
    )

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



METRICS
sari: 41.7859
bleu: 0.8326
rougeL: 0.6248
bertscore_f1: 0.9412
fkgl: 13.3727


In [18]:
OUTPUT_FILE = "GEMMA_ZERO_SHOT_ASSET_2359_FINAL.csv"

CHECKPOINT_EVERY = 25

print("Output:", OUTPUT_FILE)

Output: GEMMA_ZERO_SHOT_ASSET_2359_FINAL.csv


In [19]:
if os.path.exists(OUTPUT_FILE):

    results_df = pd.read_csv(
        OUTPUT_FILE
    )

    completed_ids = set(
        results_df["sample_id"]
        .astype(int)
    )

    print(
        f"🔄 Existing results found: "
        f"{len(completed_ids)} completed."
    )

else:

    results_df = pd.DataFrame()

    completed_ids = set()

    print(
        "🆕 Starting from sample 1."
    )

🆕 Starting from sample 1.


In [20]:
for i in range(len(asset_df)):

    # =====================================
    # RESUME
    # =====================================

    if i in completed_ids:
        continue


    # =====================================
    # DATA
    # =====================================

    example = asset_df.iloc[i]

    source = example["original"]

    references = example["simplifications"]


    # =====================================
    # PROMPT
    # =====================================

    prompt = zero_shot_prompt(
        source
    )


    # =====================================
    # TOKENIZATION
    # =====================================

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    # =====================================
    # GPU MEMORY RESET
    # =====================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    # =====================================
    # GENERATION
    # =====================================

    start_time = time.perf_counter()

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.perf_counter()


    inference_time = (
        end_time - start_time
    )


    # =====================================
    # DECODE
    # =====================================

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


    # =====================================
    # PEAK VRAM
    # IMPORTANT:
    # measured immediately after generation
    # before BERTScore
    # =====================================

    if torch.cuda.is_available():

        peak_vram = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

    else:

        peak_vram = None


    # =====================================
    # METRICS
    # =====================================

    scores = evaluate_one_sample(
        source,
        prediction,
        references
    )


    # =====================================
    # RESULT
    # =====================================

    result = {

        "sample_id": i,

        "model": "Gemma-2-2B-IT",

        "prompting": "zero-shot",

        "original": source,

        "generated": prediction,

        "references": " ||| ".join(
            references
        ),

        "sari": scores["sari"],

        "bleu": scores["bleu"],

        "rougeL": scores["rougeL"],

        "bertscore_f1": scores["bertscore_f1"],

        "fkgl": scores["fkgl"],

        "inference_time_sec": inference_time,

        "peak_vram_gb": peak_vram
    }


    # =====================================
    # APPEND
    # =====================================

    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    completed_ids.add(i)


    # =====================================
    # CHECKPOINT
    # =====================================

    if (
        len(completed_ids) % CHECKPOINT_EVERY == 0
        or i == len(asset_df) - 1
    ):

        results_df.to_csv(
            OUTPUT_FILE,
            index=False
        )

        print(
            f"\n💾 CHECKPOINT: "
            f"{len(completed_ids)}/2359 saved"
        )


    # =====================================
    # PROGRESS
    # =====================================

    print(
        f"[{i + 1}/2359] "
        f"SARI={scores['sari']:.2f} | "
        f"BLEU={scores['bleu']:.3f} | "
        f"ROUGE-L={scores['rougeL']:.3f} | "
        f"BERT={scores['bertscore_f1']:.3f} | "
        f"FKGL={scores['fkgl']:.2f} | "
        f"Time={inference_time:.2f}s | "
        f"VRAM={peak_vram:.2f}GB"
    )


    # =====================================
    # CLEANUP
    # =====================================

    del inputs
    del output_ids

    if i % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

[1/2359] SARI=41.79 | BLEU=0.833 | ROUGE-L=0.625 | BERT=0.941 | FKGL=13.37 | Time=2.83s | VRAM=5.89GB
[2/2359] SARI=39.62 | BLEU=0.751 | ROUGE-L=0.357 | BERT=0.930 | FKGL=-1.45 | Time=0.84s | VRAM=5.88GB
[3/2359] SARI=43.55 | BLEU=0.861 | ROUGE-L=0.421 | BERT=0.929 | FKGL=7.61 | Time=1.66s | VRAM=5.89GB
[4/2359] SARI=31.25 | BLEU=1.000 | ROUGE-L=0.550 | BERT=0.941 | FKGL=9.66 | Time=1.20s | VRAM=5.89GB
[5/2359] SARI=28.54 | BLEU=0.000 | ROUGE-L=0.288 | BERT=0.939 | FKGL=12.52 | Time=0.61s | VRAM=5.88GB
[6/2359] SARI=16.06 | BLEU=0.000 | ROUGE-L=0.169 | BERT=0.894 | FKGL=12.52 | Time=0.98s | VRAM=5.89GB
[7/2359] SARI=32.02 | BLEU=0.603 | ROUGE-L=0.386 | BERT=0.942 | FKGL=10.73 | Time=0.99s | VRAM=5.89GB
[8/2359] SARI=33.42 | BLEU=0.000 | ROUGE-L=0.373 | BERT=0.930 | FKGL=11.52 | Time=0.94s | VRAM=5.89GB
[9/2359] SARI=43.87 | BLEU=1.000 | ROUGE-L=0.477 | BERT=0.961 | FKGL=6.71 | Time=0.95s | VRAM=5.89GB
[10/2359] SARI=46.49 | BLEU=0.946 | ROUGE-L=0.585 | BERT=0.942 | FKGL=9.05 | Time=0.8

In [21]:
# ==========================================
# FINAL GEMMA ZERO-SHOT SUMMARY
# ==========================================

import pandas as pd
import numpy as np

df = pd.read_csv("GEMMA_ZERO_SHOT_ASSET_2359_FINAL.csv")

print("=" * 70)
print("GEMMA 2B IT — ZERO-SHOT — ASSET")
print("=" * 70)

print(f"Total samples: {len(df)}")
print()

metrics = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl",
    "inference_time_sec",
    "peak_vram_gb"
]

summary = df[metrics].agg(
    ["mean", "std", "min", "max"]
).T

summary.columns = [
    "Mean",
    "Std",
    "Min",
    "Max"
]

print(summary.round(4))

GEMMA 2B IT — ZERO-SHOT — ASSET
Total samples: 2359

                       Mean     Std      Min      Max
sari                37.4202  9.7565  13.8312  72.4922
bleu                 0.5562  0.3256   0.0000   1.0000
rougeL               0.5214  0.1562   0.0897   0.9102
bertscore_f1         0.9388  0.0220   0.8436   0.9926
fkgl                 8.4781  4.1328  -3.0100  26.4900
inference_time_sec   1.0879  0.4370   0.3073   4.6042
peak_vram_gb         5.8870  0.0020   5.8827   5.8978


In [22]:
# ==========================================
# GEMMA 2B IT — FEW-SHOT PROMPT
# ==========================================

FEW_SHOT_EXAMPLES = [
    {
        "original": "The spacecraft consists of two main elements: the NASA Cassini orbiter and the ESA Huygens probe.",
        "simplified": "The spacecraft has two main parts: the Cassini orbiter and the Huygens probe."
    },
    {
        "original": "Jeddah is the principal gateway to Mecca, Islam's holiest city.",
        "simplified": "Jeddah is the main entrance to Mecca, Islam's holiest city."
    },
    {
        "original": "His next work follows an especially eventful day in the life of a successful neurosurgeon.",
        "simplified": "His next work is about a very busy day in the life of a successful brain surgeon."
    }
]


def few_shot_prompt(sentence):

    examples_text = ""

    for example in FEW_SHOT_EXAMPLES:

        examples_text += (
            f"Original: {example['original']}\n"
            f"Simplified: {example['simplified']}\n\n"
        )

    return f"""Simplify the following sentences using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Here are some examples:

{examples_text}

Now simplify this sentence:

Original: {sentence}
Simplified:"""

In [23]:
example = asset_df.iloc[0]

source = example["original"]

prompt = few_shot_prompt(source)

print(prompt)

Simplify the following sentences using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Here are some examples:

Original: The spacecraft consists of two main elements: the NASA Cassini orbiter and the ESA Huygens probe.
Simplified: The spacecraft has two main parts: the Cassini orbiter and the Huygens probe.

Original: Jeddah is the principal gateway to Mecca, Islam's holiest city.
Simplified: Jeddah is the main entrance to Mecca, Islam's holiest city.

Original: His next work follows an especially eventful day in the life of a successful neurosurgeon.
Simplified: His next work is about a very busy day in the life of a successful brain surgeon.



Now simplify this sentence:

Original: Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).
Simplified:


In [24]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

start_time = time.perf_counter()

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

end_time = time.perf_counter()

inference_time = end_time - start_time

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

peak_vram = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
    if torch.cuda.is_available()
    else None
)

print("SOURCE:")
print(source)

print("\nGEMMA FEW-SHOT:")
print(prediction)

print("\nInference time:", round(inference_time, 3), "sec")
print("Peak VRAM:", round(peak_vram, 3), "GB")

SOURCE:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

GEMMA FEW-SHOT:
The counties bordering the area are Marin, Mendocino, Lake, Napa, Solano, and Contra Costa.

Inference time: 1.277 sec
Peak VRAM: 5.911 GB


In [25]:
test_scores = evaluate_one_sample(
    source,
    prediction,
    example["simplifications"]
)

print("\nMETRICS")
print("=" * 50)

for metric, value in test_scores.items():
    print(f"{metric}: {value:.4f}")


METRICS
sari: 42.5720
bleu: 0.7517
rougeL: 0.6595
bertscore_f1: 0.9455
fkgl: 13.4700


In [31]:
FEW_SHOT_OUTPUT = "GEMMA_FEW_SHOT_ASSET_2359_FINAL.csv"
CHECKPOINT_EVERY = 25

if os.path.exists(FEW_SHOT_OUTPUT):

    fewshot_df = pd.read_csv(FEW_SHOT_OUTPUT)

    completed_ids = set(
        fewshot_df["sample_id"].astype(int)
    )

    print(f"🔄 Existing Few-shot results: {len(completed_ids)} completed.")

else:

    fewshot_df = pd.DataFrame()
    completed_ids = set()

    print("🆕 Starting Gemma Few-shot from sample 1.")

🔄 Existing Few-shot results: 1625 completed.


In [32]:
for i in range(len(asset_df)):

    if i in completed_ids:
        continue

    example = asset_df.iloc[i]

    source = example["original"]
    references = example["simplifications"]

    prompt = few_shot_prompt(source)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.perf_counter()

    inference_time = end_time - start_time

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # Measure Gemma generation VRAM
    peak_vram = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
        if torch.cuda.is_available()
        else None
    )

    # Metrics
    scores = evaluate_one_sample(
        source,
        prediction,
        references
    )

    result = {
        "sample_id": i,
        "model": "Gemma-2-2B-IT",
        "prompting": "few-shot",
        "original": source,
        "generated": prediction,
        "references": " ||| ".join(references),
        "sari": scores["sari"],
        "bleu": scores["bleu"],
        "rougeL": scores["rougeL"],
        "bertscore_f1": scores["bertscore_f1"],
        "fkgl": scores["fkgl"],
        "inference_time_sec": inference_time,
        "peak_vram_gb": peak_vram
    }

    fewshot_df = pd.concat(
        [
            fewshot_df,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    completed_ids.add(i)

    if (
        len(completed_ids) % CHECKPOINT_EVERY == 0
        or i == len(asset_df) - 1
    ):

        fewshot_df.to_csv(
            FEW_SHOT_OUTPUT,
            index=False
        )

        print(
            f"\n💾 CHECKPOINT: "
            f"{len(completed_ids)}/2359 saved"
        )

    print(
        f"[{i + 1}/2359] "
        f"SARI={scores['sari']:.2f} | "
        f"BLEU={scores['bleu']:.3f} | "
        f"ROUGE-L={scores['rougeL']:.3f} | "
        f"BERT={scores['bertscore_f1']:.3f} | "
        f"FKGL={scores['fkgl']:.2f} | "
        f"Time={inference_time:.2f}s | "
        f"VRAM={peak_vram:.2f}GB"
    )

    del inputs
    del output_ids

    if i % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

[1626/2359] SARI=45.00 | BLEU=0.779 | ROUGE-L=0.633 | BERT=0.938 | FKGL=6.88 | Time=2.06s | VRAM=5.91GB
[1627/2359] SARI=59.18 | BLEU=0.951 | ROUGE-L=0.725 | BERT=0.967 | FKGL=10.50 | Time=1.69s | VRAM=5.91GB
[1628/2359] SARI=37.29 | BLEU=0.740 | ROUGE-L=0.614 | BERT=0.939 | FKGL=25.00 | Time=2.99s | VRAM=5.91GB
[1629/2359] SARI=53.61 | BLEU=0.851 | ROUGE-L=0.640 | BERT=0.955 | FKGL=3.65 | Time=1.07s | VRAM=5.91GB
[1630/2359] SARI=39.67 | BLEU=0.678 | ROUGE-L=0.584 | BERT=0.935 | FKGL=18.58 | Time=1.51s | VRAM=5.91GB
[1631/2359] SARI=0.00 | BLEU=0.000 | ROUGE-L=0.000 | BERT=0.000 | FKGL=0.00 | Time=0.19s | VRAM=5.91GB
[1632/2359] SARI=39.68 | BLEU=0.695 | ROUGE-L=0.667 | BERT=0.933 | FKGL=11.75 | Time=1.89s | VRAM=5.91GB
[1633/2359] SARI=39.35 | BLEU=0.597 | ROUGE-L=0.455 | BERT=0.930 | FKGL=7.37 | Time=0.74s | VRAM=5.91GB
[1634/2359] SARI=31.17 | BLEU=0.768 | ROUGE-L=0.497 | BERT=0.929 | FKGL=12.84 | Time=1.89s | VRAM=5.91GB
[1635/2359] SARI=59.59 | BLEU=1.000 | ROUGE-L=0.606 | BERT=0

In [33]:
# ==========================================
# GEMMA 2B IT — FEW-SHOT FINAL SUMMARY
# ASSET — 2359 SOURCES
# ==========================================

import pandas as pd
import numpy as np
import os

FEW_SHOT_OUTPUT = "GEMMA_FEW_SHOT_ASSET_2359_FINAL.csv"

# ------------------------------------------
# Load final CSV
# ------------------------------------------

fewshot_df = pd.read_csv(FEW_SHOT_OUTPUT)

print("=" * 70)
print("GEMMA 2B IT — FEW-SHOT — ASSET FINAL EVALUATION")
print("=" * 70)

print(f"Total rows: {len(fewshot_df)}")

# ------------------------------------------
# Check sample IDs
# ------------------------------------------

unique_ids = fewshot_df["sample_id"].nunique()

expected_ids = set(range(2359))
actual_ids = set(
    fewshot_df["sample_id"].astype(int)
)

missing_ids = expected_ids - actual_ids
duplicate_count = len(fewshot_df) - unique_ids

print(f"Unique sample IDs: {unique_ids}")
print(f"Duplicate rows: {duplicate_count}")
print(f"Missing sample IDs: {len(missing_ids)}")

if missing_ids:
    print("Missing IDs:", sorted(missing_ids)[:20])
else:
    print("✅ No missing sample IDs")

# ------------------------------------------
# Empty generations
# ------------------------------------------

empty_mask = (
    fewshot_df["generated"]
    .fillna("")
    .astype(str)
    .str.strip()
    == ""
)

empty_count = empty_mask.sum()
empty_percentage = (
    empty_count / len(fewshot_df)
) * 100

print()
print(f"Empty generations: {empty_count}")
print(f"Empty generation rate: {empty_percentage:.2f}%")

# ------------------------------------------
# Metric summary
# ------------------------------------------

metrics = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl",
    "inference_time_sec",
    "peak_vram_gb"
]

summary = pd.DataFrame({
    "Mean": fewshot_df[metrics].mean(),
    "Std": fewshot_df[metrics].std(),
    "Min": fewshot_df[metrics].min(),
    "Max": fewshot_df[metrics].max()
})

print()
print("=" * 70)
print("FEW-SHOT METRIC SUMMARY")
print("=" * 70)

display(summary.round(4))

# ------------------------------------------
# Save summary
# ------------------------------------------

summary.to_csv(
    "GEMMA_FEW_SHOT_ASSET_2359_SUMMARY.csv"
)

print()
print("✅ Summary saved:")
print("GEMMA_FEW_SHOT_ASSET_2359_SUMMARY.csv")

# ------------------------------------------
# Basic integrity check
# ------------------------------------------

if (
    len(fewshot_df) == 2359
    and unique_ids == 2359
    and duplicate_count == 0
    and len(missing_ids) == 0
):
    print()
    print("🎉 DATASET INTEGRITY CHECK PASSED")
    print("2359/2359 unique ASSET sources successfully evaluated.")
else:
    print()
    print("⚠️ DATASET INTEGRITY CHECK NEEDS REVIEW")

GEMMA 2B IT — FEW-SHOT — ASSET FINAL EVALUATION
Total rows: 2359
Unique sample IDs: 2359
Duplicate rows: 0
Missing sample IDs: 0
✅ No missing sample IDs

Empty generations: 1
Empty generation rate: 0.04%

FEW-SHOT METRIC SUMMARY


,Mean,Std,Min,Max
sari,41.3967,9.2367,0.0000,80.2756
bleu,0.6563,0.2599,0.0000,1.0000
rougeL,0.5772,0.1371,0.0000,0.9560
bertscore_f1,0.9473,0.0269,0.0000,0.9928
fkgl,8.4199,3.5770,-2.2300,25.0000
inference_time_sec,1.2067,0.4348,0.1915,3.7951
peak_vram_gb,5.9098,0.0018,5.9063,5.9212



✅ Summary saved:
GEMMA_FEW_SHOT_ASSET_2359_SUMMARY.csv

🎉 DATASET INTEGRITY CHECK PASSED
2359/2359 unique ASSET sources successfully evaluated.
